# MPI sweep (real) — v11, 27.10

Copy of `MPI_sweep_real_v11.ipynb` with the GUI parameters of 27.10 (last GUI run `Results/gui/2026-10-27_111855_test_frame_000011`) and the approach changes of commit 135b373 (A1 restricted to AB peaks, `SIGMA_U_FLOOR` 0.1, `SIGMA_TOF_SCALE` 4):

- peak selection: the unified local-bias rule of `approaches.py` (no `AB_SELECT_MODE` any more), `FUSION_MPI_SIGMA = False`
- 33x33: auto SL tilt correction (`sl_tilt_correction`, fitted to the 10x10 corner), so 33x33 and 10x10 agree on the fold distance; 10x10 uncorrected
- scene GT: refitted after the tilt step from the 10x10 **and** the tilt-corrected 33x33 SL points (distance level from the 10x10, wall shape from ~10× more dots)
- caches in `Results/mpi_sweep_real_v11_27.10/` (the cache names do not encode the new approach settings); the PCDs/EXRs of step 1 are reused


# MPI Sweep (real) — corner reconstruction: 10x10 vs. 33x33 vs. flood

The real counterpart of `MPIsweep.ipynb`: the same concave corner (vertical fold) measured with the
Schmersal AM-T100, once per illumination:

| set | VCSEL | delay (coarse/fine/superfine) | exposure |
|---|---|---|---|
| `10x10` | 5 (10x10 DOE) | 7 / 0 / 5 | 100 µs |
| `33x33` | 6 (coherent 33x33) | 0 / 0 / 0 | 100 µs |
| `Flood` | 4 (flood) | 7 / 0 / 5 | 1000 µs |

The corners are labelled by their **inner angle**. The real target could not reproduce the simulated
angles exactly, so the angles differ from `MPIsweep.ipynb`, and there is only one fold distance per
measurement series. Corner and camera did not move between the three illuminations of one angle.
Two series, selected with `MEAS_ROOT`:

| series | fold distance | inner angles |
|---|---|---|
| `MPI_Messung_07.10/` | ≈ 1.3 m | 45°, 60°, 90° (outputs archived in `MPI_sweep_real_1.3m.html`) |
| `MPI_Messung_07.10/MPI_1m_WW/` | ≈ 1 m | 30°, 45°, 60°, 90° |

Data: `<MEAS_ROOT>/<date>_<time>_<pattern>_<angle>/ToF_RAW/frame_*.npy`,
10 frames per set, AandB (8 tap images per frame), 100 MHz. The notebook turns the RAWs into an
organised PCD + an intensity EXR per set (step 1) — by default from the **first frame** of every set
(`FRAME_MODE`), as the calibration also used single frames. Then it runs Approach 1/2/3 with the
existing real calibrations, fits the ground-truth corner from the data, and draws the plots of
`MPIsweep.ipynb`. All heavy lifting lives in `dot_calibration.py`, `approaches.py` and
`itofcameracalibration/tof_processing`; this notebook only orchestrates and plots.

In [ ]:
from dot_calibration import DotCalibration
import approaches
import calibrate
import contextlib
import io
import os
import re
import sys
import numpy as np
from pathlib import Path
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
import matplotlib.pyplot as plt
import logging
from scipy.ndimage import map_coordinates
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # GPR subpixel fits, harmless
logging.getLogger("matplotlib.axes._base").setLevel(logging.ERROR)   # "Ignoring fixed x limits …" (equal aspect)

dotCal = DotCalibration()

# ── Data locations ──────────────────────────────────────────────────────────
# MPI_Messung_07.10 (fold ≈ 1.3 m, 45/60/90°) or MPI_Messung_07.10/MPI_1m_WW (fold ≈ 1 m, 30/45/60/90°)
MEAS_ROOT     = Path(os.environ.get("MPI_MEAS_ROOT", "/home/julian/Documents/Calibrations/MPI_Messung_07.10/MPI_1m_WW"))
ITOF_CAL_REPO = Path("/home/julian/Documents/Calibrations/itofcameracalibration_new/itofcameracalibration")   # tof_processing (v1.1 HDF5)
HDF5_ROOT     = Path("/home/julian/Documents/Calibrations")
ANGLES        = (30, 45, 60, 90)      # inner angle of the corner [deg]; incomplete angles are skipped
PATTERNS      = {"10x10": dict(label="10x10"), "33x33": dict(label="33x33")}
FLOOD         = "Flood"

# Depth calibration per pattern. It has to match the one the dot calibration's PCDs were made
# with — otherwise the ToF range is off by a constant per VCSEL/delay setting (see step 1):
#   10x10 (25.08, delay 7/0/5)    → CalibData_Julian.hdf5        (Pos3: 530 mm vs. 532 mm in the PCD)
#   33x33 (30.09, delay 0/0/0)    → CalibData_Julian_LensOnly.hdf5 (Pos3: 575 mm vs. 573 mm)
#   Flood (delay 7/0/5, like 10x10) → CalibData_Julian.hdf5
# v11: dot-sweep calibrations (itofcameracalibration_new, dots mode). Flood keeps its old HDF5 and the
# PCDs already written with the old pipeline (OUT_DIRS below), since there is no new flood calibration.
HDF5 = {"10x10": HDF5_ROOT / "CalibData_10x10_v1.1.hdf5",
        "33x33": HDF5_ROOT / "CalibData_Coherent_v1.1.hdf5",
        FLOOD:   HDF5_ROOT / "CalibData_Julian.hdf5"}

# Settings every set must have been recorded with (taken from the calibration sessions).
REF_SETTINGS = {"10x10": dict(ActiveVcsel=0b010000, DelayCoarse=7, DelayFine=0, DelaySuperFine=5, ExposureTime=100),
                "33x33": dict(ActiveVcsel=0b100000, DelayCoarse=0, DelayFine=0, DelaySuperFine=0, ExposureTime=100),
                FLOOD:   dict(ActiveVcsel=0b001000, DelayCoarse=7, DelayFine=0, DelaySuperFine=5, ExposureTime=1000)}

# Existing real calibrations (codePaperlike), no recalibration here.
#   "corr13456": iToF_SL_fusion_pipeline.ipynb with the 6-stage ToF error model (stages 1–6 on);
#                the test clouds get the same model (cell "ToF error model" below)
#   "lpc":       the earlier calibrations, light-path correction only
#   "v11":       SL calibrations from the v1.1 ToF calibration (noiseEstimation_v11_*), no LPC, no error model
CAL_VARIANT = "v11"
CAL_ROOT = Path.cwd().parent / "Calibrations"
CAL_SETS = {
    "corr13456": {"10x10": "SchmersalReal_paperlike_sor6-0.40_lpc_corr13456_06.10_",
                  "33x33": "SchmersalRealCoherent_paperlike_sor6-0.40_lpc_corr13456_06.10_CoherentLens"},
    "lpc":       {"10x10": "SchmersalReal_paperlike_sor6-0.40_lpc_06.10",
                  "33x33": "SchmersalRealCoherent_paperlike_sor6-0.40_lpc_06.10_CoherentLens"},
    "v11":       {"10x10": "SchmersalReal_v11_noise_sor6-1.00_cap0.00-4.00",
                  "33x33": "SchmersalRealCoherent_v11_noise_sor6-1.00_cap0.00-4.00"},
}
CAL_FILES = {pat: CAL_ROOT / name / "calibration.json" for pat, name in CAL_SETS[CAL_VARIANT].items()}
# SL tilt correction (sl_tilt_correction.py): the Coherent projector is not rigidly mounted, a small
# rotation after the calibration shifts its whole depth scale. Per pattern:
#   None                                           no correction
#   dict(mode="manual", offset_mm=90, z_ref=1.1)   the SL reads 90 mm too far at 1.1 m (e.g. vs. 10x10)
#   dict(mode="manual", tilt_mrad=3.3)             the tilt directly
#   dict(mode="auto")                              estimated from the scene GT (10x10 corner), see the tilt step
#   dict(mode="z_shift", z_shift_mm=37)            constant z shift of the trail labels (as in the GUI)
SL_TILT = {"10x10": None, "33x33": dict(mode="auto")}   # 33x33 aligned to the 10x10 corner, see the tilt step
TILT_TAG = {}                         # cache tag per pattern, set in the tilt step
# Wall measurement the error model of each pattern is fitted on (keys of WALL_SETS below).
ERR_CAMERA = {"10x10": "SchmersalReal", "33x33": "SchmersalRealCoherent"}

F_MOD          = 100                  # [MHz]
UNAMBIGUITY    = 299792458.0 / (2 * F_MOD * 1e6)   # [m] 1.499 m
Z_MAX          = 1.45                 # [m] keep clear of the phase wrap
FRAME_MODE     = "first"              # "first": first RAW frame of every set | "avg": 10-frame average
WRITE_FRAMES   = False                # "avg" only: also write every single frame as PCD + EXR

CACHE_DIR   = Path.cwd().parent / "Results" / "mpi_sweep_real_v11_27.10"
# Output folders per pattern inside each set (v11 must not overwrite the old PCDs; flood reuses them)
OUT_DIRS    = {"10x10": ("ToF_PCD_v11", "SL_v11"), "33x33": ("ToF_PCD_v11", "SL_v11"), FLOOD: ("ToF_PCD", "SL")}
FORCE_RERUN = False                   # True → redo RAW conversion and sweep, ignore caches
CACHE_DIR.mkdir(parents=True, exist_ok=True)

if str(ITOF_CAL_REPO) not in sys.path:
    sys.path.insert(0, str(ITOF_CAL_REPO))
from tof_processing.tof_processing import tof_pipeline, parse_DeltaRawData

approaches.GT_DISTANCE = None         # no single GT distance in a corner scene
# GUI parameters 27.10 (everything else: defaults of approaches.py)
approaches.FUSION_MPI_SIGMA = False
print("Setup complete.")

## Measurement sets

Every folder `<date>_<time>_<pattern>_<angle>` is one set; if a (pattern, angle) was recorded more
than once, the newest folder wins. The camera YAML stored next to the RAWs is checked against the
settings of the calibration sessions (VCSEL, delays, exposure) — a mismatch is printed as a warning,
because a different delay shifts the ToF range by a constant (≈ 0.2–0.6 m here).

In [ ]:
SET_RE = re.compile(r"^(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})_(10x10|33x33|Flood)_(\d+)$")
YAML_KEYS = ("ActiveVcsel", "DelayCoarse", "DelayFine", "DelaySuperFine", "ExposureTime",
             "Frequency", "SensorOutputMode", "CustomVcselPower", "calibrationFile")


def read_camera_yaml(set_dir):
    # The camera YAML holds binary literals (0b010000) and comments → read the keys by regex.
    out = {}
    for y in Path(set_dir).glob("*.yaml"):
        txt = re.sub(r"#.*", "", y.read_text())
        for k in YAML_KEYS:
            m = re.search(rf"^\s*{k}\s*:\s*\"?([^\"\n]+?)\"?\s*$", txt, re.M)
            if m:
                v = m.group(1).strip()
                out[k] = int(v, 0) if re.fullmatch(r"0b[01]+|\d+", v) else v
    return out


SETS = {}
for d in sorted(MEAS_ROOT.iterdir()):
    m = SET_RE.match(d.name)
    if not (d.is_dir() and m):
        continue
    stamp, pat, angle = m.group(1), m.group(2), int(m.group(3))
    frames = sorted((d / "ToF_RAW").glob("frame_*.npy"))
    if not frames:
        print(f"skip {d.name}: no RAW frames")
        continue
    if (pat, angle) in SETS:
        print(f"{pat} {angle}°: {SETS[(pat, angle)]['dir'].name} replaced by the newer {d.name}")
    SETS[(pat, angle)] = dict(dir=d, frames=frames, yaml=read_camera_yaml(d))

print(f"{'set':34s} {'frames':>6s}  VCSEL     delay   exp[µs]  power  mode    calibrationFile")
for (pat, angle), s in sorted(SETS.items(), key=lambda kv: (kv[0][1], kv[0][0])):
    y = s["yaml"]
    print(f"{s['dir'].name:34s} {len(s['frames']):6d}  {y.get('ActiveVcsel', 0):#08b}  "
          f"{y.get('DelayCoarse')}/{y.get('DelayFine')}/{y.get('DelaySuperFine')}  "
          f"{y.get('ExposureTime'):>7}  {y.get('CustomVcselPower'):>5}  {y.get('SensorOutputMode'):6s}  "
          f"{y.get('calibrationFile')}")
    bad = {k: (y.get(k), v) for k, v in REF_SETTINGS[pat].items() if y.get(k) != v}
    if bad or y.get("SensorOutputMode") != "AandB":
        print(f"    WARNING {pat} {angle}°: differs from the calibration session: "
              + ", ".join(f"{k} {a} (ref {b})" for k, (a, b) in bad.items()))

ANGLES = tuple(a for a in ANGLES if all((p, a) in SETS for p in list(PATTERNS) + [FLOOD]))
print(f"\ncomplete corners (10x10 + 33x33 + flood): {ANGLES}")

## Step 1 — RAW → PCD + intensity EXR

Per set, `FRAME_MODE = "first"` takes the first RAW frame (`"avg"`: the 10 frames averaged tap by tap,
A and B separately) and it goes through `tof_pipeline` with the HDF5 of its pattern (phase offset, FPPN, wiggling, lens
model). The averaged taps are passed as A − B differences (`process_2rad(Diffs=…)`):
`process_fromRaw` would cast the averaged taps to int32 and throw away the sub-LSB part that the
averaging gained. The intensity (A + B)/2 is `calc_intensity` of the same averaged taps. This is
exactly what `binary2pcd.py` writes, just for the new folder layout:

- `<set>/ToF_PCD/<frame>.pcd` (or `avg.pcd`) — organised 640×480, fields `x y z grayValue rad distZ` [mm]
- `<set>/SL/<frame>.exr` (or `avg.exr`) — intensity in channel `S0.940,000nm` (like `pcdToExr.py`), the SL image of
  the approaches

The pixel rays `(x/z, y/z)` of the lens model are kept: the approaches return pixel + axial depth,
and turning that into x, y with the same rays as the clouds keeps dots and flood in one frame.

In [ ]:
import OpenEXR
import Imath

PIPES = {str(h): tof_pipeline(fMod=F_MOD, height=480, width=640, calibrationFile=str(h))
         for h in sorted(set(map(str, HDF5.values())))}


def pipe_rays(pipe):
    # Lens-model rays: X = Z·rx, Y = Z·ry (tof_pipeline.calc_XfromZ / calc_YfromZ).
    rx = (pipe.lens_u_corr - pipe.lens_c_mtx[0, 2]) / pipe.lens_c_mtx[0, 0]
    ry = (pipe.lens_v_corr - pipe.lens_c_mtx[1, 2]) / pipe.lens_c_mtx[1, 1]
    return np.asarray(rx, float), np.asarray(ry, float)


def process_raw(pipe, raw):
    # One AandB frame (single or averaged) → X, Y, Z, rad, intensity [mm / LSB].
    A, B = parse_DeltaRawData(raw)
    A, B = np.asarray(A, np.float64), np.asarray(B, np.float64)
    pipe.process_2rad(Diffs=list(A - B))
    pipe.intensity = pipe.calc_intensity(A, B)
    pipe.calc_XYZ()
    return [np.asarray(a, np.float64) for a in (pipe.X, pipe.Y, pipe.Z, pipe.radDist, pipe.intensity)]


def write_pcd(path, X, Y, Z, gray, rad):
    fields = ("x", "y", "z", "grayValue", "rad", "distZ")
    data = np.empty(X.size, dtype=[(f, "<f8") for f in fields])
    for f, a in zip(fields, (X, Y, Z, gray, rad, Z)):
        data[f] = np.asarray(a, np.float64).ravel()
    H, W = X.shape
    header = ("# .PCD v.7 - Point Cloud Data file format\nVERSION .7\n"
              f"FIELDS {' '.join(fields)}\nSIZE {' '.join(['8'] * 6)}\nTYPE {' '.join(['F'] * 6)}\n"
              f"COUNT {' '.join(['1'] * 6)}\nWIDTH {W}\nHEIGHT {H}\nVIEWPOINT 0 0 0 1 0 0 0\n"
              f"POINTS {X.size}\nDATA binary\n")
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        f.write(header.encode("ascii"))
        f.write(data.tobytes())


def write_exr(path, img):
    h, w = img.shape
    header = OpenEXR.Header(w, h)
    header["channels"] = {dotCal.SL_CHANNEL: Imath.Channel(Imath.PixelType(Imath.PixelType.FLOAT))}
    path.parent.mkdir(parents=True, exist_ok=True)
    out = OpenEXR.OutputFile(str(path), header)
    out.writePixels({dotCal.SL_CHANNEL: np.ascontiguousarray(img, np.float32).tobytes()})
    out.close()


def convert_set(pat, s):
    name = s["frames"][0].stem if FRAME_MODE == "first" else "avg"
    pcd_dir, sl_dir = (s["dir"] / d for d in OUT_DIRS[pat])
    pcd, exr = pcd_dir / f"{name}.pcd", sl_dir / f"{name}.exr"
    s.update(pcd=pcd, exr=exr, frame=name)
    if pcd.exists() and exr.exists() and (not FORCE_RERUN or pat == FLOOD):
        return False
    pipe = PIPES[str(HDF5[pat])]
    if FRAME_MODE == "first":
        X, Y, Z, rad, inten = process_raw(pipe, np.load(s["frames"][0]).astype(np.float64))
        write_pcd(pcd, X, Y, Z, inten, rad)
        write_exr(exr, inten)
        return True
    raws = np.array([np.load(f).astype(np.float64) for f in s["frames"]])
    X, Y, Z, rad, inten = process_raw(pipe, raws.mean(axis=0))
    write_pcd(pcd, X, Y, Z, inten, rad)
    write_exr(exr, inten)
    if WRITE_FRAMES:
        for f, raw in zip(s["frames"], raws):
            X, Y, Z, rad, inten = process_raw(pipe, raw)
            write_pcd(pcd_dir / f"{f.stem}.pcd", X, Y, Z, inten, rad)
            write_exr(sl_dir / f"{f.stem}.exr", inten)
    return True


for (pat, angle), s in sorted(SETS.items()):
    new = convert_set(pat, s)
    print(f"{s['dir'].name:34s} → {s['pcd'].parent.name}/{s['frame']}.pcd + {s['exr'].parent.name}/{s['frame']}.exr  ({'written' if new else 'cached'}, "
          f"{Path(HDF5[pat]).name})")

FRAME_TAG = "_first" if FRAME_MODE == "first" else "_avg"
RAYS = {pat: pipe_rays(PIPES[str(HDF5[pat])]) for pat in list(PATTERNS) + [FLOOD]}
_r = [np.stack(r) for r in RAYS.values()]
print(f"\nlens rays of the two HDF5 files differ by at most "
      f"{max(np.nanmax(np.abs(a - _r[0])) for a in _r) * 505:.4f} px")

### Check: pipeline, intrinsics and raw data

- The processed cloud must follow the lens model of the calibration PCDs: a pinhole fit
  u = f_x·x/z + c_x over the averaged cloud gives f_x ≈ 495 px (`30.09/ToF`: 494.9, `25.08/ToF`: 495.9;
  without lens model it would be ≈ 1074).
- Noise: the per-pixel range scatter over the 10 single frames, binned by amplitude, and what is left
  after averaging (÷√10).
- Overview per set: intensity (the SL image), axial depth, top view of the central rows.

In [ ]:
def fit_K_from_pcd(pcd_path):
    t = dotCal.load_tof_pcd(str(pcd_path), unit_scale=0.001, depth_mode="axial")
    P = t["points_map"]
    H, W = P.shape[:2]
    v, u = np.mgrid[0:H, 0:W]
    ok = np.isfinite(P).all(axis=-1)
    x, y, z = P[..., 0][ok], P[..., 1][ok], P[..., 2][ok]
    (fx_, cx_), *_ = np.linalg.lstsq(np.c_[x / z, np.ones_like(x)], u[ok], rcond=None)
    (fy_, cy_), *_ = np.linalg.lstsq(np.c_[y / z, np.ones_like(y)], v[ok], rcond=None)
    return fx_, fy_, cx_, cy_


def frame_noise(pat, s):
    # Range scatter over the single frames [mm] per amplitude bin (circular std of the phase).
    I, Q = [], []
    for f in s["frames"]:
        A, B = (np.asarray(x, float) for x in parse_DeltaRawData(np.load(f).astype(np.float64)))
        D = A - B
        I.append(D[0] - D[2]); Q.append(D[3] - D[1])
    z = np.array(I) + 1j * np.array(Q)
    amp = np.abs(z.mean(axis=0))
    dphi = np.angle(z * np.exp(-1j * np.angle(z.mean(axis=0))))
    sig = np.sqrt(np.mean(dphi ** 2, axis=0)) * UNAMBIGUITY * 1e3 / (2 * np.pi)
    edges = (20, 50, 150, 500, 1e9)
    return [(lo, float(np.median(sig[(amp >= lo) & (amp < hi)])) if ((amp >= lo) & (amp < hi)).sum() > 200 else np.nan)
            for lo, hi in zip(edges[:-1], edges[1:])]


print(f"{'set':34s}  f_x     f_y     c_x    c_y   | σ_range single frame [mm] for amplitude ≥ 20 / 50 / 150 / 500 LSB")
for (pat, angle), s in sorted(SETS.items(), key=lambda kv: (kv[0][1], kv[0][0])):
    fx_, fy_, cx_, cy_ = fit_K_from_pcd(s["pcd"])
    nz = frame_noise(pat, s)
    print(f"{s['dir'].name:34s} {fx_:6.1f}  {fy_:6.1f}  {cx_:6.1f} {cy_:6.1f} | "
          + "  ".join("   –  " if np.isnan(v) else f"{v:6.1f}" for _, v in nz))
if FRAME_MODE == "avg":
    print(f"(10 frames averaged → ≈ ÷{np.sqrt(10):.1f})")

In [ ]:
OVERVIEW_ROWS = slice(200, 280)     # rows of the top view


def load_set(pat, angle):
    return dotCal.load_tof_pcd(str(SETS[(pat, angle)]["pcd"]), unit_scale=0.001, depth_mode="axial")


cols = list(PATTERNS) + [FLOOD]
fig, axs = plt.subplots(len(ANGLES), 3 * len(cols), figsize=(3.0 * 3 * len(cols), 2.6 * len(ANGLES)), squeeze=False)
for i, angle in enumerate(ANGLES):
    for j, pat in enumerate(cols):
        t = load_set(pat, angle)
        g, Pm = t["intensity_map"], t["points_map"]
        a0, a1, a2 = axs[i, 3 * j], axs[i, 3 * j + 1], axs[i, 3 * j + 2]
        a0.imshow(g, cmap="gray", vmin=np.nanpercentile(g, 1), vmax=np.nanpercentile(g, 99.7))
        a1.imshow(np.clip(Pm[..., 2], 0, Z_MAX), cmap="viridis")
        band = Pm[OVERVIEW_ROWS].reshape(-1, 3)
        band = band[np.isfinite(band).all(axis=1)]
        a2.scatter(band[:, 0], band[:, 2], s=0.2, lw=0, alpha=0.3)
        a2.set_xlim(-0.9, 0.9); a2.set_ylim(0, 1.6); a2.set_aspect("equal"); a2.grid(alpha=0.3)
        a2.axhline(UNAMBIGUITY, color="r", lw=0.6, ls=":")
        for a, t_ in zip((a0, a1, a2), ("intensity", "z (axial)", "top view")):
            a.set_title(f"{pat} {angle}° · {t_}", fontsize=8)
            a.tick_params(labelsize=6)
        a0.axis("off"); a1.axis("off")
fig.suptitle("Averaged sets (red dotted: unambiguity range 1.5 m — the background behind the corner wraps)", y=1.0)
fig.tight_layout()
plt.show()

## Statistical outlier removal (as in the calibration)

The dot calibrations (`…_sor6-0.40_…`) and `iToF_SL_fusion_pipeline.ipynb` filter every ToF cloud with
CloudCompare's SOR before the light-path correction and the error model: a point is rejected when the
mean distance to its `SOR_K` nearest neighbours exceeds mean + `SOR_NSIGMA`·std over the cloud. The
same filter (`sor_keep_mask`, copied from the fusion notebook) runs here on every loaded cloud — dot
series and flood — in memory: rejected points become NaN, the files on disk stay untouched. It
matters more for single frames than for the 10-frame average.

In [ ]:
# ── SOR: CloudCompare statistical outlier removal (iToF_SL_fusion_pipeline.ipynb) ──
from scipy.spatial import cKDTree

SOR_ENABLED = True
SOR_K       = 6       # CloudCompare "points to use for mean distance estimation"
SOR_NSIGMA  = 0.4     # CloudCompare "standard deviation multiplier" (as in the calibrations)
SOR_TAG     = f"_sor{SOR_K}-{SOR_NSIGMA:.2f}" if SOR_ENABLED else ""
SOR_LOG     = {}      # pcd path → (valid, kept)


def sor_keep_mask(xyz, k=None, nsigma=None, z_range=None):
    # CloudCompare's statistical outlier removal (copied from iToF_SL_fusion_pipeline.ipynb).
    # Returns (keep, threshold, n_valid); pixels without a return come back as not-kept.
    k = int(SOR_K if k is None else k)
    nsigma = float(SOR_NSIGMA if nsigma is None else nsigma)
    z_lo, z_hi = DotCalibration.VALID_Z_RANGE if z_range is None else z_range

    xyz = np.asarray(xyz, dtype=np.float64)
    valid = (np.isfinite(xyz).all(axis=1)
             & np.any(xyz != 0.0, axis=1)
             & (xyz[:, 2] >= z_lo) & (xyz[:, 2] <= z_hi))
    keep = np.zeros(len(xyz), dtype=bool)
    idx = np.flatnonzero(valid)
    if idx.size <= k + 1:                      # too few points to say anything
        keep[idx] = True
        return keep, float("inf"), int(idx.size)

    pts = xyz[idx]
    d, _ = cKDTree(pts).query(pts, k=k + 1, workers=1)
    mean_d = d[:, 1:].mean(axis=1)
    thr = float(mean_d.mean() + nsigma * mean_d.std())
    keep[idx] = mean_d <= thr
    return keep, thr, int(idx.size)


def _sor_load_tof_pcd(self, pcd_path, *args, **kwargs):
    t = _sor_inner_load_tof_pcd(self, pcd_path, *args, **kwargs)
    if not SOR_ENABLED:
        return t
    depth_mode = kwargs.get("depth_mode", args[1] if len(args) > 1 else "radial")
    keep, _, n_valid = sor_keep_mask(t["points_3d"])
    P = t["points_3d"].copy()
    P[~keep] = np.nan
    SOR_LOG[str(pcd_path)] = (n_valid, int(keep.sum()))
    t["points_3d"] = P
    t["distance"] = P[:, 2] if depth_mode in ("z", "axial") else np.linalg.norm(P, axis=1)
    if "points_map" in t:
        H, W = t["points_map"].shape[:2]
        t["points_map"] = P.reshape(H, W, 3)
        t["depth_map"] = t["distance"].reshape(H, W)
    return t


_sor_load_tof_pcd._is_sor = True
if not getattr(DotCalibration.load_tof_pcd, "_is_sor", False) and not getattr(DotCalibration.load_tof_pcd, "_is_lpc", False):
    _sor_inner_load_tof_pcd = DotCalibration.load_tof_pcd
    DotCalibration.load_tof_pcd = _sor_load_tof_pcd
print(f"SOR {'ON' if SOR_ENABLED else 'OFF'}: {SOR_K} neighbours, nSigma = {SOR_NSIGMA:.2f}  (tag '{SOR_TAG}')")


## ToF light-path correction

As in `MPIsweep.ipynb`: the dot projector (VCSEL 5 / 6) also lights the ToF, from the baseline
**B** instead of the lens centre, so the camera reports m = (|P − B| + |P|)/2. The calibrations were
made with this correction (`_lpc`), and `tof_light_origin` stores the B they used; the same B corrects
the test clouds here: **r = (4m² − |B|²) / (4m − 2 d̂·B)**, P = r·d̂. The flood (VCSEL 4) has no
calibration and no known B; its cloud stays uncorrected (see the ground-truth step).

In [ ]:
LIGHT_PATH_CORRECTION = False   # v11: projector offset is in the FPPN of the ToF calibration
LPC_TAG     = "_lpc" if LIGHT_PATH_CORRECTION else ""
TOF_LIGHT_B = None            # [m] light source in the PCD frame, set per pattern while loading
TOF_ERR_MODEL = None          # fitted ToF error model (stages 1, 3–6), set per pattern while loading


def correct_light_path(P, B):
    # True points for ToF points P (N, 3) [m] measured with the light source at B.
    P = np.asarray(P, dtype=np.float64)
    B = np.asarray(B, dtype=np.float64)
    with np.errstate(invalid="ignore", divide="ignore"):
        m = np.linalg.norm(P, axis=1)
        d = P / m[:, None]
        r = (4.0 * m * m - B @ B) / (4.0 * m - 2.0 * (d @ B))
    return d * r[:, None]


def _lpc_load_tof_pcd(self, pcd_path, *args, **kwargs):
    # Light-path correction (stage 2), then the fitted error model (stages 1, 3–6) — the same
    # order as in iToF_SL_fusion_pipeline.ipynb.
    t = _lpc_inner_load_tof_pcd(self, pcd_path, *args, **kwargs)
    if TOF_LIGHT_B is None and TOF_ERR_MODEL is None:
        return t
    depth_mode = kwargs.get("depth_mode", args[1] if len(args) > 1 else "radial")
    P = t["points_3d"]
    if TOF_LIGHT_B is not None:
        P = correct_light_path(P, TOF_LIGHT_B)
    H, W = t["points_map"].shape[:2] if "points_map" in t else (None, None)
    P = correct_tof_model(P, H, W)
    t["points_3d"] = P
    t["distance"] = P[:, 2] if depth_mode in ("z", "axial") else np.linalg.norm(P, axis=1)
    if "points_map" in t:
        t["points_map"] = P.reshape(H, W, 3)
        t["depth_map"] = t["distance"].reshape(H, W)
    return t


_lpc_load_tof_pcd._is_lpc = True
if not getattr(DotCalibration.load_tof_pcd, "_is_lpc", False):
    _lpc_inner_load_tof_pcd = DotCalibration.load_tof_pcd
DotCalibration.load_tof_pcd = _lpc_load_tof_pcd

# ── Calibrations ────────────────────────────────────────────────────────────
CALS = {}
for pat, f in CAL_FILES.items():
    cal = approaches.load_calibration(str(f))
    CALS[pat] = cal
    md_ = cal.get("metadata", {})
    B_ = md_.get("tof_light_origin")
    print(f"{pat}: {f.parent.name}\n    created {md_.get('created', '?')[:10]}, "
          f"{sum(t is not None for t in cal['trails'])} trails, mode {cal.get('mode')}, "
          f"B = {np.round(np.asarray(cal['B']) * 1e3, 2).tolist()} mm, "
          f"light origin = {None if B_ is None else np.round(np.asarray(B_) * 1e3, 2).tolist()}")


def light_origin(pat):
    if not LIGHT_PATH_CORRECTION or pat not in CALS:
        return None
    o = CALS[pat].get("metadata", {}).get("tof_light_origin")
    return None if o is None else np.asarray(o, dtype=float)

## ToF error model (6 correction stages)

`CAL_VARIANT = "corr13456"` uses the calibrations of `iToF_SL_fusion_pipeline.ipynb` with all
correction stages on. Those calibrations were made on ToF clouds that had already gone through the
error model, so the test clouds here get **the same model** — otherwise B, V, the trails and the test
depths would not match:

| Stage | Correction | Physical cause |
|---|---|---|
| 1 | constant offset | zero point / emitter delay |
| 2 | light-path correction (LPC) | light source at B instead of the lens (cell above) |
| 3 | image gradient, constant in mm | effective light origin ≠ B, pixel offsets |
| 4 | image gradient ∝ Z | optical axis of the r→Z conversion |
| 5 | wiggling (periods R/4, R/2, R) | cyclic iToF error, not calibrated for the projector |
| 6 | fixed error per dot | dot-specific residual (organised clouds only) |

`calibration.json` stores only the terms of stages 1, 3–5, not the per-dot bias of stage 6, so the
model is refitted here on the same flat-wall data with the code of the fusion notebook (copied
verbatim, only `correct_tof_model` takes the model as an argument). The refitted terms are compared
with the ones stored in the calibration.

Validity: same processing chain and projector as the wall data — 10x10 ↔ `CalibData_Julian.hdf5`,
33x33 ↔ `CalibData_Julian_LensOnly.hdf5`, exactly the HDF5 files of step 1. Stage 6 only acts where a
pixel lies within 3 px of a wall-session dot **and** at a similar inverse depth (|Δw| < 0.1 m⁻¹); in
the corner the dots sit at other depths than on the wall, so stage 6 applies only partly. The flood
(VCSEL 4) has no wall data and stays uncorrected.

In [ ]:
# ── ToF error model: switchable correction stages (fitted on the flat-wall data) ──
# Copied from iToF_SL_fusion_pipeline.ipynb (cell "ToF error model").
import csv
from scipy.spatial import cKDTree

CORR_STAGES = {1: True, 2: True, 3: True, 4: True, 5: True, 6: True}

# Wall measurement per camera (binary2pcd_manual.py output: dots/<d>m.csv) and the B used
# for the LPC while fitting (the pipeline's own LPC iteration keeps estimating B itself).
WALL_SETS = {
    "SchmersalReal": dict(
        root=Path("/home/julian/Documents/Calibrations/RauschmessungManualUnwrapping"),
        exclude=(0.50, 0.55, 0.60, 0.65, 0.70),   # overwritten 01.10 with a coherent dot calibration
        B_ref=[0.73e-3, -39.07e-3, 33.64e-3]),
    "SchmersalRealCoherent": dict(
        root=Path("/home/julian/Documents/Calibrations/RauschmessungManualUnwrappingCoherentLensCalib"),
        exclude=(),
        B_ref=[-20.48e-3, -39.61e-3, 14.68e-3]),
}
CORR_K = (530.7, 530.68, 322.78, 242.17)   # fx, fy, cx, cy of the lens calibration the PCDs use
CORR_MAX_DIST = 2.0      # [m] beyond this the 100 MHz unwrapping fails too often
CORR_GROSS = 0.15        # [m] wrong-k outliers left out of the fit
CORR_DOT_RADIUS = 3.0    # [px] stage 6: pixel ↔ dot assignment radius
CORR_DOT_DW = 0.1        # [1/m] stage 6: inverse-depth tolerance of that assignment
_CORR_R = 299_792_458.0 / (2 * 100e6)       # unambiguity range at 100 MHz
_CORR_W_SCALE = CORR_DOT_RADIUS / CORR_DOT_DW


def _corr_lpc(P, B):
    # Same formula as correct_light_path in the light-path cell (defined further down).
    B = np.asarray(B, dtype=np.float64)
    m = np.linalg.norm(P, axis=1)
    d = P / m[:, None]
    return d * ((4.0 * m * m - B @ B) / (4.0 * m - 2.0 * (d @ B)))[:, None]


def _corr_features(xn, yn, Z, r, stages):
    cols, names = [np.ones_like(Z)], ["offset"]
    if stages[3]:
        cols += [xn, yn]; names += ["grad_x", "grad_y"]
    if stages[4]:
        cols += [xn * Z, yn * Z]; names += ["gradZ_x", "gradZ_y"]
    if stages[5]:
        for T, lab in ((_CORR_R / 4, "R/4"), (_CORR_R / 2, "R/2"), (_CORR_R, "R")):
            cols += [np.sin(2 * np.pi * r / T), np.cos(2 * np.pi * r / T)]
            names += [f"sin {lab}", f"cos {lab}"]
    return np.column_stack(cols), names


def fit_tof_error_model(camera, stages):
    # Fit all enabled stages jointly on the wall dots of `camera`. None if no wall data.
    ws = WALL_SETS.get(camera)
    if ws is None:
        return None
    fxc, fyc, cxc, cyc = CORR_K
    rows = []
    for f in ws["root"].glob("dots/*m.csv"):
        d = float(re.search(r"([\d.]+)m\.csv$", f.name).group(1))
        if d > CORR_MAX_DIST or any(abs(d - e) < 1e-6 for e in ws["exclude"]):
            continue
        for r in csv.DictReader(open(f)):
            if r["valid"] == "1" and r["median_Z_mm"] != "nan":
                rows.append((d, int(r["id"]), float(r["u_sub"]), float(r["v_sub"]),
                             float(r["median_Z_mm"]) * 1e-3))
    D, ids, u, v, Z = (np.array(c) for c in zip(*rows))
    ids = ids.astype(int)
    xn, yn = (u - cxc) / fxc, (v - cyc) / fyc
    P = np.c_[xn * Z, yn * Z, Z]
    if stages[2]:
        P = _corr_lpc(P, ws["B_ref"])
    Zc = P[:, 2]
    e_raw = Z - D
    ok = np.abs(Zc - D) < CORR_GROSS
    F, names = _corr_features(xn, yn, Zc, np.linalg.norm(P, axis=1), stages)
    coef, *_ = np.linalg.lstsq(F[ok], (Zc - D)[ok], rcond=None)
    apply = coef.copy()
    if not stages[1]:
        apply[0] = 0.0                      # offset fitted (keeps the other terms unbiased), not applied
    e_fit = Zc - D - F @ apply
    model = dict(camera=camera, stages=dict(stages), names=names, coef=coef, apply=apply,
                 root=str(ws["root"]), n=int(ok.sum()), n_dots=len(np.unique(ids[ok])))
    if stages[6]:
        res = (Zc - D - F @ coef)[ok]
        dot_bias = {i: float(np.median(res[ids[ok] == i])) for i in np.unique(ids[ok])}
        model["tree"] = cKDTree(np.c_[u[ok], v[ok], _CORR_W_SCALE / D[ok]])
        model["obs_bias"] = np.array([dot_bias[i] for i in ids[ok]])
        e_fit = e_fit - np.where(ok, np.array([dot_bias.get(i, 0.0) for i in ids]), 0.0)
    rms = lambda e: float(np.sqrt(np.mean(e[ok] ** 2)) * 1e3)
    model["rms_before"], model["rms_after"] = rms(e_raw), rms(e_fit)
    return model


def correct_tof_model(P, H=None, W=None, model=None):
    # Subtract the fitted error model from points P (N, 3) [m], along each ray.
    m = TOF_ERR_MODEL if model is None else model
    if m is None:
        return P
    P = np.asarray(P, dtype=np.float64)
    Z = P[:, 2]
    with np.errstate(invalid="ignore", divide="ignore"):
        if H is not None and W is not None and H * W == len(P):
            fxc, fyc, cxc, cyc = CORR_K
            vv, uu = np.mgrid[0:H, 0:W]
            u, v = uu.ravel().astype(float), vv.ravel().astype(float)
            xn, yn = (u - cxc) / fxc, (v - cyc) / fyc
        else:
            u = v = None
            xn, yn = P[:, 0] / Z, P[:, 1] / Z
        F, _ = _corr_features(xn, yn, Z, np.linalg.norm(P, axis=1), m["stages"])
        e = F @ m["apply"]
        if "tree" in m and u is not None:
            good = np.isfinite(Z) & (Z > 0)
            dist, j = m["tree"].query(np.c_[u[good], v[good], _CORR_W_SCALE / Z[good]])
            obs = m["tree"].data[j]
            hit = (np.hypot(obs[:, 0] - u[good], obs[:, 1] - v[good]) < CORR_DOT_RADIUS) \
                & (np.abs(obs[:, 2] - _CORR_W_SCALE / Z[good]) < CORR_DOT_RADIUS)
            e6 = np.zeros(int(good.sum()))
            e6[hit] = m["obs_bias"][j[hit]]
            e[good] += e6
        return P * ((Z - e) / Z)[:, None]


ERR_MODELS = {}
if CAL_VARIANT == "corr13456":
    for pat in PATTERNS:
        m = fit_tof_error_model(ERR_CAMERA[pat], CORR_STAGES)
        ERR_MODELS[pat] = m
        stored = (CALS[pat].get("metadata", {}).get("error_model") or {}).get("terms", {})
        diff = max((abs(m["apply"][i] - stored[n]) for i, n in enumerate(m["names"]) if n in stored),
                   default=np.nan)
        print(f"{pat}: model of {ERR_CAMERA[pat]} from {m['n']} wall-dot measurements ({m['n_dots']} dots), "
              f"wall RMS {m['rms_before']:.1f} → {m['rms_after']:.1f} mm")
        print("    " + "   ".join(f"{n} {c * 1e3:+.2f}" for n, c in zip(m["names"], m["apply"])) + "   [mm]")
        print(f"    max |refit − stored term| = {diff * 1e3:.4f} mm"
              + ("" if diff < 1e-5 else "   WARNING: differs from the model the calibration was made with"))
CORR_TAG = "_corr13456" if ERR_MODELS else ""
print(f"error model {'ON' if ERR_MODELS else 'OFF'} (cache tag '{LPC_TAG}{CORR_TAG}')")

## Sweep: Approach 1/2/3 on every corner

Per (corner, pattern): one `common_setup` (averaged EXR + PCD) feeds all three runners, like
`run_dots` in `MPIsweep.ipynb`. The approaches work along the calibrated dot trails, so no dot
detection is needed at test time. Each valid dot becomes a 3-D point from its pixel position and
axial depth:

| | depth | pixel |
|---|---|---|
| Approach 1 – consistency error | `Z_out` (ToF at min ε) | `u_out, v_out` |
| Approach 2 – ML fusion | `z_fus` | `u_sub, v_sub` |
| Approach 3 – active brightness | `Z_out` (triangulated) | `u_sub, v_sub` |

X = r_x(u, v)·Z, Y = r_y(u, v)·Z with the lens-model rays of step 1, the same as in the clouds.
The flood is the raw iToF cloud (points beyond `Z_MAX` dropped, they are wrapped background).
Results are cached in `Results/mpi_sweep_real/` (`<pattern>_<angle>deg_<set>….npz`).

In [ ]:
APPROACH_KEYS = {1: ("Z_out", "u_out", "v_out"), 2: ("z_fus", "u_sub", "v_sub"), 3: ("Z_out", "u_sub", "v_sub")}


def to_xyz(results, zkey, ukey, vkey, rays):
    rows = [(r[ukey], r[vkey], r[zkey]) for r in results
            if np.isfinite(r.get(zkey, np.nan)) and np.isfinite(r.get(ukey, np.nan))]
    if not rows:
        return np.zeros((0, 3))
    u, v, z = np.array(rows, float).T
    rx = map_coordinates(rays[0], [v, u], order=1, mode="nearest")
    ry = map_coordinates(rays[1], [v, u], order=1, mode="nearest")
    return np.c_[rx * z, ry * z, z]


def run_dots(pat, angle):
    global TOF_LIGHT_B, TOF_ERR_MODEL
    s = SETS[(pat, angle)]
    cache = CACHE_DIR / f"{pat}_{angle}deg_{s['dir'].name}{FRAME_TAG}{SOR_TAG}{LPC_TAG}{CORR_TAG}{TILT_TAG.get(pat, '')}{'_mpisig' if approaches.FUSION_MPI_SIGMA else ''}.npz"
    if cache.exists() and not FORCE_RERUN:
        with np.load(cache) as f:
            return {a: f[f"A{a}"] for a in (1, 2, 3)} | {"n_trails": int(f["n_trails"])}
    cal = CALS[pat]
    TOF_LIGHT_B = light_origin(pat)               # same corrections as in the calibration
    TOF_ERR_MODEL = ERR_MODELS.get(pat)
    try:
        with contextlib.redirect_stdout(io.StringIO()):
            setup = approaches.common_setup(dotCal, cal, str(s["exr"]), str(s["pcd"]))
            res = {1: approaches.run_approach_1(dotCal, cal, setup, make_plots=False),
                   2: approaches.run_approach_2(dotCal, cal, setup, make_plots=False),
                   3: approaches.run_approach_3(dotCal, cal, setup, make_plots=False)}
    finally:
        TOF_LIGHT_B, TOF_ERR_MODEL = None, None
    out = {a: to_xyz(res[a]["results"], *APPROACH_KEYS[a], RAYS[pat]) for a in (1, 2, 3)}
    out["n_trails"] = sum(t is not None for t in setup["trails"])
    np.savez_compressed(cache, **{f"A{a}": out[a] for a in (1, 2, 3)}, n_trails=out["n_trails"])
    return out


def load_flood(angle):
    # Flood cloud as organised map (H, W, 3) [m], wrapped / invalid points NaN.
    P = load_set(FLOOD, angle)["points_map"].copy()
    P[~(P[..., 2] < Z_MAX)] = np.nan
    return P


RAW = {}        # (angle, series, approach or "ToF") → (N, 3) points, camera frame [m]
N_TRAILS = {}
FLOOD_MAP = {}
for angle in ANGLES:
    for pat in PATTERNS:
        r = run_dots(pat, angle)
        N_TRAILS[pat] = r["n_trails"]
        for a in (1, 2, 3):
            RAW[(angle, pat, a)] = r[a]
        print(f"{angle}° {pat}: valid dots A1/A2/A3 = {len(r[1])}/{len(r[2])}/{len(r[3])} of {r['n_trails']} trails")
    FLOOD_MAP[angle] = load_flood(angle)
    print(f"{angle}° flood: {np.isfinite(FLOOD_MAP[angle][..., 2]).sum()} points")
if SOR_ENABLED:
    print("SOR rejected: " + ", ".join(f"{Path(p).parent.parent.name.split('_', 2)[-1]} "
                                       f"{100 * (1 - k / max(v, 1)):.1f} %" for p, (v, k) in SOR_LOG.items()))

## Ground truth: one corner per angle, from the SL triangulation (10x10, then 10x10 + tilt-corrected 33x33)

There is no scene file, so the corner is fitted from the data. Camera and target stay put while the
illumination changes, so there is **one** ground truth per angle for all series (10x10, 33x33, flood).
Multipath only ever makes the ToF range *longer*, so the GT must not depend on the ToF depth:

1. **Flood walls (frame and region only):** the fold column is the column with the largest median depth
   in the central rows (searched in the image centre). RANSAC planes (inliers < `RANSAC_THR`) to the
   flood pixels left and right of the fold, from `FOLD_MARGIN` (measured along the wall) up to
   `WALL_WIDTH_PX` columns away. They define the region of interest and the start of the wall split.
2. **Scene GT:** the 10x10 dots triangulated from the SL image alone (`sl_points`): per trail the most
   prominent brightness peak with a 2-D dot nearby — the 10x10 trails are unique, so no ToF is needed
   to pick it — refined on the trail as in `approaches.py`. RANSAC planes to these points farther than
   the margin from the fold give the corner. Every series is shown in this corner frame: y along the
   fold, z from the camera to the fold, x across; the fold sits at (0, d).
3. **Walls end at `WALL_LENGTH`** = 0.69 m from the fold (the real wall), so RMSE refers to the target.
4. **Region of interest** = an image region: the pixels whose flood point lies on its wall, within the
   y band ≤ ±`Y_HALF`, the x range up to the wall edges / image border and at most `WALL_LENGTH` from
   the fold. Every point of every series is assigned to its pixel by its ray direction and kept if that
   pixel is in the region.

5. **Refit after the tilt step:** the 10x10 has only ~40–60 dots on the walls, too few for a stable
   corner at 1.3 m (at 90° its fold landed ~8 cm beside the fold of the flood and the 33x33, and the
   flood lay in front of it). Once the 33x33 trails are tilt-corrected to the 10x10 corner, the GT is
   refitted from the SL points of both patterns: the distance level stays the 10x10's (that is what the
   tilt is fitted to), the wall orientation and fold position come from ~10× more dots.

Why not the uncorrected 33x33 triangulation (the previous GT, from each pattern's own A3): in this session it puts
the walls 9–13 cm farther than the 10x10 (a near-constant offset of ≈ 0.05–0.08 1/m in 1/z, ≈ 1.3 px
along the trails), with mostly a single peak per trail — not a wrong-peak choice. With it, the 33x33 ToF
would lie *in front of* the GT, which multipath cannot do; the 10x10 corner agrees with the target's
wall length (0.69 m from the image edges) and the tape. On the flat-wall sweep of 30.09/01.10 the
33x33 triangulation was accurate to ≈ 1 mm, so its calibration does not fit this session (07.10).


In [ ]:
WALL_LENGTH     = 0.69    # [m] fold → outer edge, horizontal (0.68–0.70 m); GT walls end there
RANSAC_THR      = 0.015   # [m]
FOLD_MARGIN     = 0.15    # [m] wall points used for the fits keep this distance to the fold, measured
                          # ALONG the wall (laterally: FOLD_MARGIN·sin(inner/2), so narrow corners keep wall left)
FOLD_SEARCH_U   = (220, 420)   # [px] columns searched for the fold
WALL_WIDTH_PX   = 170     # [px] wall band used for the flood fit, beyond the margin
FIT_ROWS        = slice(120, 360)
MIN_WALL_POINTS = 10
Y_HALF          = 0.40    # [m] max. half height of the evaluated band on the walls
ROI_TOL         = 0.05    # [m] a flood point "lies on the wall" within this distance
_rng_fit = np.random.default_rng(1)


def fit_plane(P):
    c = P.mean(axis=0)
    n = np.linalg.svd(P - c, full_matrices=False)[2][2]
    d = -n @ c
    return (n, d) if d > 0 else (-n, -d)          # normal towards the camera (origin on + side)


def ransac_plane(P, thr=RANSAC_THR, iters=400):
    best = None
    for _ in range(iters):
        p = P[_rng_fit.choice(len(P), 3, replace=False)]
        n = np.cross(p[1] - p[0], p[2] - p[0])
        if np.linalg.norm(n) < 1e-12:
            continue
        n /= np.linalg.norm(n)
        inl = np.abs((P - p[0]) @ n) < thr
        if best is None or inl.sum() > best.sum():
            best = inl
    for _ in range(2):
        n, d = fit_plane(P[best])
        best = np.abs(P @ n + d) < thr
    return n, d, best


def flood_walls(Pmap, margin_x):
    zc = np.nanmedian(Pmap[FIT_ROWS, :, 2], axis=0)
    zc_s = np.convolve(np.nan_to_num(zc), np.ones(9) / 9, mode="same")
    u_fold = FOLD_SEARCH_U[0] + int(np.argmax(zc_s[FOLD_SEARCH_U[0]:FOLD_SEARCH_U[1]]))
    margin_px = margin_x * 495.0 / zc_s[u_fold]
    band = Pmap[FIT_ROWS]
    u = np.broadcast_to(np.arange(band.shape[1]), band.shape[:2])
    ok = np.isfinite(band).all(axis=-1)
    walls, share = [], []
    for side in ((u < u_fold - margin_px) & (u > u_fold - margin_px - WALL_WIDTH_PX),
                 (u > u_fold + margin_px) & (u < u_fold + margin_px + WALL_WIDTH_PX)):
        n, d, inl = ransac_plane(band[ok & side])
        walls.append((n, d))
        share.append(inl.mean())
    return walls, u_fold, share


def ray_range_to_corner(P, walls):
    # Range along the ray of every point P to the (concave) corner = nearer wall intersection.
    dh = P / np.linalg.norm(P, axis=1, keepdims=True)
    t = np.stack([-d / (dh @ n) for n, d in walls], axis=1)
    t[t <= 0] = np.inf
    return t.min(axis=1)


def corner_frame(walls):
    (n1, d1), (n2, d2) = walls
    ey = np.cross(n1, n2); ey /= np.linalg.norm(ey)
    if ey[1] < 0:
        ey = -ey
    O = np.linalg.solve(np.array([n1, n2, ey]), np.array([-d1, -d2, 0.0]))   # fold point nearest the camera
    ez = O - (O @ ey) * ey; ez /= np.linalg.norm(ez)
    ex = np.cross(ey, ez)
    if ex[0] < 0:
        ex = -ex
    R = np.stack([ex, ey, ez])
    dist = float(np.linalg.norm(O))
    shift = np.array([0.0, 0.0, dist])
    to_corner = lambda P: (np.asarray(P) - O) @ R.T + shift
    return dict(to_corner=to_corner, R=R, O=O, dist=dist, walls=walls,
                corner=corner_polyline(walls, R, O, shift))


def corner_polyline(walls, R, O, shift):
    # Walls (camera frame) → top-view polyline [(x_left, z_left), (x_fold, z_fold), (x_right, z_right)]
    # in the corner frame, each wall running WALL_LENGTH from the fold towards the camera.
    lines = []
    for n, d in walls:
        nc = R @ n
        lines.append((nc[0], nc[2], d + n @ O - nc @ shift))      # a·x + b·z + c = 0  (y = 0)
    (a1, b1, c1), (a2, b2, c2) = lines
    xf, zf = np.linalg.solve([[a1, b1], [a2, b2]], [-c1, -c2])
    ends = []
    for a, b, _ in lines:
        t = np.array([b, -a]) / np.hypot(a, b)                       # direction along the wall
        t = t if t[1] < 0 else -t                                     # … towards the camera
        ends.append((xf + WALL_LENGTH * t[0], zf + WALL_LENGTH * t[1]))
    c = np.array([ends[0], [xf, zf], ends[1]])
    return c[np.argsort(c[:, 0])]


def inner_angle(corner):
    v = [(c - corner[1]) / np.linalg.norm(c - corner[1]) for c in corner[[0, 2]]]
    return float(np.degrees(np.arccos(np.clip(v[0] @ v[1], -1, 1))))


def z_gt(x, corner):
    # GT depth of the corner at lateral position x (single-valued over the corner width).
    return np.interp(x, corner[:, 0], corner[:, 1], left=np.nan, right=np.nan)


def dist_to_gt(x, z, corner):
    # Perpendicular distance [m] of points (x, z) to the nearer of the two walls.
    pts = np.c_[x, z]
    best = np.full(len(pts), np.inf)
    for a, b in ((corner[0], corner[1]), (corner[1], corner[2])):
        ab = b - a
        t = np.clip(((pts - a) @ ab) / (ab @ ab), 0, 1)
        best = np.minimum(best, np.linalg.norm(pts - (a + t[:, None] * ab), axis=1))
    return best


def lateral_margin(corner):
    # FOLD_MARGIN along the wall → lateral distance from the fold.
    return FOLD_MARGIN * np.sin(np.radians(inner_angle(corner) / 2))


GT = {}          # flood walls per angle: frame for the region of interest
for angle in ANGLES:
    mx = FOLD_MARGIN * np.sin(np.radians(angle / 2))          # start from the nominal angle …
    for _ in range(3):                                         # … then from the fitted one
        walls, u_fold, share = flood_walls(FLOOD_MAP[angle], mx)
        GT[angle] = corner_frame(walls)
        mx = lateral_margin(GT[angle]["corner"])
    GT[angle].update(mx=mx, u_fold=u_fold)
    print(f"{angle}° flood walls: fold column {u_fold}, fold at d = {GT[angle]['dist']:.3f} m, inner angle "
          f"{inner_angle(GT[angle]['corner']):.1f}°  (RANSAC inliers {share[0]:.0%} / {share[1]:.0%}, "
          f"lateral margin {mx * 100:.1f} cm)")

In [ ]:
# ── Scene GT: one corner per angle from the 10x10 triangulation alone (no ToF depth) ──
import json
import sl_tilt_correction
FOLD_DISTANCE = {"MPI_1m_WW": 1.03}   # [m] tape-measured camera–fold distance (information only)
GT_PATTERN = "10x10"                  # its trails are unique: one dot per trail, no ToF needed to pick it
GT_OF = {"10x10": GT_PATTERN, "33x33": GT_PATTERN, FLOOD: GT_PATTERN}   # every series → the scene corner


def sl_points(pat, angle, cal=None):
    # Triangulated dot points [m, camera frame] of one set from the SL image only: per trail the most
    # prominent brightness peak that has a 2-D dot (blob) nearby, refined like approaches.ab_trail_candidates
    # (sl_tilt_correction.sl_dot_peaks). The ToF depth is not used anywhere, so the points carry no multipath.
    s = SETS[(pat, angle)]
    with contextlib.redirect_stdout(io.StringIO()):
        setup = approaches.common_setup(dotCal, CALS[pat] if cal is None else cal, str(s["exr"]), str(s["pcd"]))
    pk = sl_tilt_correction.sl_dot_peaks(dotCal, setup)
    if not len(pk["w"]):
        return np.zeros((0, 3))
    z = 1.0 / pk["w"]
    rx = map_coordinates(RAYS[pat][0], [pk["v"], pk["u"]], order=1, mode="nearest")
    ry = map_coordinates(RAYS[pat][1], [pk["v"], pk["u"]], order=1, mode="nearest")
    P = np.c_[rx * z, ry * z, z]
    return P[P[:, 2] < Z_MAX]


def series_walls(P_cam, frame):
    # Two walls from one series: RANSAC per side on its points far from the fold (None if too few).
    Pc = frame["to_corner"](P_cam)
    walls = []
    for side in (Pc[:, 0] < -frame["mx"], Pc[:, 0] > frame["mx"]):
        if side.sum() < MIN_WALL_POINTS:
            return None, f"{side.sum()} wall points"
        n, d, _ = ransac_plane(P_cam[side], thr=max(RANSAC_THR, 0.02), iters=400)
        walls.append((n, d))
    return walls, ""


def fit_shape(P_cam, frame):
    # Walls from P_cam, the margin iterated with the walls' own frame (the flood frame is only the start).
    for _ in range(3):
        walls, why = series_walls(P_cam, frame)
        if walls is None:
            return None, why
        frame = corner_frame(walls)
        frame["mx"] = lateral_margin(frame["corner"])
    return walls, ""


def wall_spread(P_cam, frame):
    # Robust spread [mm] of the GT points around their own walls, far from the fold.
    Pc = frame["to_corner"](P_cam)
    far = np.abs(Pc[:, 0]) > frame["mx"]
    e = dist_to_gt(Pc[far, 0], Pc[far, 2], frame["corner"])
    return 1.4826 * float(np.median(e)) * 1e3, int(far.sum())


SCENE, PGT, GT_PTS = {}, {}, {}
tape = FOLD_DISTANCE.get(MEAS_ROOT.name)
print(f"Scene GT per angle from the {GT_PATTERN} triangulation (SL only, no ToF): fold distance d, inner angle"
      + (f"  — tape-measured fold distance {tape:.3f} m (reference point on the camera unknown)" if tape else ""))
for angle in ANGLES:
    P = sl_points(GT_PATTERN, angle)
    w, why = fit_shape(P, GT[angle])
    if w is None:
        raise RuntimeError(f"{angle}°: {GT_PATTERN} SL points give no corner ({why})")
    fr = corner_frame(w)
    fr["mx"] = lateral_margin(fr["corner"])
    SCENE[angle], GT_PTS[angle] = fr, P
    for pat in PATTERNS:
        PGT[(angle, pat)] = fr
    spread, n_far = wall_spread(P, fr)
    # for comparison: the 33x33 triangulation (same SL-only peak choice) and the previous GT (own A3)
    w33, _ = fit_shape(sl_points("33x33", angle), GT[angle])
    w33a3, _ = fit_shape(RAW[(angle, "33x33", 3)], GT[angle])
    d33 = f"{corner_frame(w33)['dist']:.3f}" if w33 else "–"
    d33a3 = f"{corner_frame(w33a3)['dist']:.3f}" if w33a3 else "–"
    print(f"  {angle}°: d = {fr['dist']:.3f} m, {inner_angle(fr['corner']):5.1f}°  ({len(P)} dots, "
          f"{n_far} on the walls, spread {spread:.1f} mm)   |   33x33 SL d = {d33} m, "
          f"33x33 A3 (previous GT) d = {d33a3} m, flood walls d = {GT[angle]['dist']:.3f} m")

GT_EXPORT = CACHE_DIR / f"gt_scene_{MEAS_ROOT.name}.json"
GT_EXPORT.write_text(json.dumps({str(a): dict(walls=[(n.tolist(), float(d)) for n, d in SCENE[a]["walls"]],
                                              dist=SCENE[a]["dist"], inner=inner_angle(SCENE[a]["corner"]),
                                              mx=SCENE[a]["mx"], n_dots=len(GT_PTS[a]))
                                 for a in ANGLES}, indent=1))
print(f"scene GT → {GT_EXPORT}")


## SL tilt correction (33x33 projector)

The Coherent projector is not rigidly mounted. A small rotation after the calibration moves every dot
by the same few pixels along its trail, i.e. shifts the 1/z labels of the trails by f·δ / gain
(`sl_tilt_correction.py`). In `"auto"` mode δ is estimated per angle from the scene GT (the 33x33 dot
peaks are compared with the 10x10 corner along their pixel rays, walls far from the fold only) and the
median over the angles is applied; the leave-one-angle-out column shows how well one δ carries over to
an angle it was not fitted on. `"manual"` takes the offset seen against the 10x10 (or δ) from `SL_TILT`.
The approaches of that pattern are then rerun with the corrected trails.


In [ ]:
def z_scene(angle, pat):
    # Axial depth of the scene corner along the pixel ray (u, v) of `pat`; NaN off the walls / near the fold.
    fr = SCENE[angle]
    cap = WALL_LENGTH * np.sin(np.radians(inner_angle(fr["corner"]) / 2))

    def f(u, v):
        u, v = np.atleast_1d(u).astype(float), np.atleast_1d(v).astype(float)
        r = np.c_[map_coordinates(RAYS[pat][0], [v, u], order=1, mode="nearest"),
                  map_coordinates(RAYS[pat][1], [v, u], order=1, mode="nearest"), np.ones(len(u))]
        t = np.stack([-d / (r @ n) for n, d in fr["walls"]], axis=1)
        t[t <= 0] = np.inf
        z = t.min(axis=1)
        x = fr["to_corner"](r * np.where(np.isfinite(z), z, 0.0)[:, None])[:, 0]
        return np.where((np.abs(x) > fr["mx"]) & (np.abs(x) < cap) & (z < Z_MAX), z, np.nan)
    return f


def scene_dist(P, angle):
    # Fold distance of the corner fitted to SL points P — nan if they give no corner.
    w, _ = fit_shape(P, GT[angle])
    return corner_frame(w)["dist"] if w else np.nan


TILT = {}
for pat, cfg in SL_TILT.items():
    if not cfg:
        continue
    cal0 = CALS[pat]
    mode_ = cal0.get("mode", "y")
    if cfg["mode"] == "z_shift":
        CALS[pat] = sl_tilt_correction.apply_z_shift(cal0, cfg["z_shift_mm"])
        TILT_TAG[pat] = f"_zshift{cfg['z_shift_mm']:+.0f}"
        print(f"{pat}: SL z shift — trail depth labels − {cfg['z_shift_mm']:.1f} mm")
        for angle in ANGLES:
            r = run_dots(pat, angle)
            for a in (1, 2, 3):
                RAW[(angle, pat, a)] = r[a]
            print(f"   {angle}° {pat} rerun with the shifted trails: valid dots A1/A2/A3 = "
                  f"{len(r[1])}/{len(r[2])}/{len(r[3])}")
        print("   corrected SL corner d / scene d: " + ", ".join(
            f"{a}°: {scene_dist(sl_points(pat, a), a):.3f} / {SCENE[a]['dist']:.3f} m" for a in ANGLES))
        continue
    if cfg["mode"] == "manual":
        tilt = cfg.get("tilt_mrad")
        if tilt is None:
            tilt = sl_tilt_correction.tilt_from_offset(cal0["trails"], cal0["K"], mode_,
                                                       cfg["offset_mm"], cfg["z_ref"])["tilt_mrad"]
        print(f"{pat}: manual — " + sl_tilt_correction.describe(tilt, cal0["K"], mode_, cal0["trails"]))
    else:
        per = {}
        for angle in ANGLES:
            s = SETS[(pat, angle)]
            with contextlib.redirect_stdout(io.StringIO()):
                setup = approaches.common_setup(dotCal, cal0, str(s["exr"]), str(s["pcd"]))
            per[angle] = sl_tilt_correction.estimate_tilt(dotCal, [(setup, z_scene(angle, pat))], cal0["K"], mode_)
        tilt = float(np.median([e["tilt_mrad"] for e in per.values()]))
        print(f"{pat}: tilt from the scene GT ({GT_PATTERN} corner) per angle, and leave-one-angle-out check")
        print(f"   {'angle':>5s} {'δ [mrad]':>9s} {'Δp [px]':>8s} {'MAD [px]':>9s} {'dots':>5s} {'⊥ trail':>9s}"
              f" | {'d scene':>8s} {'d SL raw':>9s} {'d SL LOO':>9s} {'LOO − scene':>12s}")
        for angle, e in per.items():
            loo = float(np.median([per[a]["tilt_mrad"] for a in per if a != angle]))
            d_raw = scene_dist(sl_points(pat, angle, cal0), angle)
            d_loo = scene_dist(sl_points(pat, angle, sl_tilt_correction.apply_tilt(cal0, loo, "loo")), angle)
            print(f"   {angle:4d}° {e['tilt_mrad']:+9.3f} {e['dpx']:+8.2f} {e['dpx_mad']:9.2f} {e['n']:5d} "
                  f"{e['d_perp_med']:7.2f}px | {SCENE[angle]['dist']:8.3f} {d_raw:9.3f} {d_loo:9.3f} "
                  f"{(d_loo - SCENE[angle]['dist']) * 1e3:+10.1f}mm")
        print(f"{pat}: auto — " + sl_tilt_correction.describe(tilt, cal0["K"], mode_, cal0["trails"]))
    TILT[pat] = tilt
    CALS[pat] = sl_tilt_correction.apply_tilt(cal0, tilt, cfg["mode"])
    TILT_TAG[pat] = f"_tilt{tilt:+.2f}"
    for angle in ANGLES:
        r = run_dots(pat, angle)
        for a in (1, 2, 3):
            RAW[(angle, pat, a)] = r[a]
        print(f"   {angle}° {pat} rerun with the tilted trails: valid dots A1/A2/A3 = "
              f"{len(r[1])}/{len(r[2])}/{len(r[3])}")
    print("   corrected SL corner d / scene d: " + ", ".join(
        f"{a}°: {scene_dist(sl_points(pat, a), a):.3f} / {SCENE[a]['dist']:.3f} m" for a in ANGLES))


### Scene GT refit: 10x10 + tilt-corrected 33x33

The tilt step aligned the 33x33 trails to the 10x10 corner, so both patterns now triangulate the same
fold distance and their SL points (no ToF) can be fitted together. The table compares the corners and
checks the GT against the flood: multipath only lengthens the ToF range, so the flood should lie
**behind** the GT everywhere (columns: median flood − GT range per lateral band [mm], + = behind;
the flood itself carries an unknown constant offset of a few mm).

Then every series gets its own GT, the scene corner turned slightly about its fold and scaled about
the camera (same shape, slightly different orientation and distance). 33x33: scale only, from its own SL
points. 10x10: turn + scale fitted to its own SL points. Flood: the turn from the left/right asymmetry of
flood − GT (multipath is symmetric about the fold, a turn is not), and one range offset per series (not
per angle): the largest one that keeps the flood behind its GT in every lateral band of every angle, so it
touches the widest corner, where the multipath is smallest.


In [ ]:
GT_SOURCES = ("10x10", "33x33")   # SL points the final scene GT is fitted to (33x33 only if tilt-corrected)
FLOOD_BANDS = ((-0.5, -0.2), (-0.2, -0.05), (-0.03, 0.03), (0.05, 0.2), (0.2, 0.5))   # [m] lateral, corner frame


def flood_behind(walls, angle):
    # Median flood range − GT range along the ray [mm] per lateral band (+ = flood behind the GT).
    fr = corner_frame(walls)
    fl = FLOOD_MAP[angle].reshape(-1, 3)
    fl = fl[np.isfinite(fl).all(axis=1)]
    x, y = fr["to_corner"](fl)[:, :2].T
    dd = np.linalg.norm(fl, axis=1) - ray_range_to_corner(fl, walls)
    ok = (np.abs(y) < 0.25) & (np.abs(dd) < 0.15)          # on the walls, background dropped
    return [float(np.median(dd[m])) * 1e3 if (m := ok & (x > lo) & (x < hi)).sum() > 50 else np.nan
            for lo, hi in FLOOD_BANDS]


sources = [p for p in GT_SOURCES if p == "10x10" or p in TILT or TILT_TAG.get(p)]
print(f"Scene GT refit from the SL points of {' + '.join(sources)}; flood − GT per lateral band "
      + " | ".join(f"{lo:+.2f}…{hi:+.2f}" for lo, hi in FLOOD_BANDS) + " m  [mm, + = flood behind]")
for angle in ANGLES:
    P = np.concatenate([sl_points(p, angle) for p in sources])
    w, why = fit_shape(P, GT[angle])
    if w is None:
        raise RuntimeError(f"{angle}°: SL points give no corner ({why})")
    old = SCENE[angle]
    fr = corner_frame(w)
    fr["mx"] = lateral_margin(fr["corner"])
    SCENE[angle], GT_PTS[angle] = fr, P
    for pat in PATTERNS:
        PGT[(angle, pat)] = fr
    spread, _ = wall_spread(P, fr)
    fold_dx = (fr["O"] - old["O"]) @ old["R"][0] * 1e3
    fmt = lambda v: " ".join(f"{x:+5.0f}" for x in v)
    print(f"  {angle}°: d = {fr['dist']:.3f} m, {inner_angle(fr['corner']):5.1f}° ({len(P)} dots, spread {spread:.1f} mm)"
          f"  — 10x10 only: d = {old['dist']:.3f} m, {inner_angle(old['corner']):5.1f}°, fold {fold_dx:+.0f} mm beside")
    print(f"        flood − GT  new: {fmt(flood_behind(w, angle))}   10x10 only: {fmt(flood_behind(old['walls'], angle))}")

GT_EXPORT.write_text(json.dumps({str(a): dict(walls=[(n.tolist(), float(d)) for n, d in SCENE[a]["walls"]],
                                              dist=SCENE[a]["dist"], inner=inner_angle(SCENE[a]["corner"]),
                                              mx=SCENE[a]["mx"], n_dots=len(GT_PTS[a]))
                                 for a in ANGLES}, indent=1))
print(f"scene GT → {GT_EXPORT}")


# ── Per-series GT: the scene corner, turned about its fold and scaled about the camera, per series ──
# 33x33: scale only (s = median range ratio of its own SL points on the walls) — it defines the scene shape.
# 10x10: turn φ about the fold axis + scale, fitted robustly to its own SL points (both are SL only).
# Flood (no SL): φ from the left/right asymmetry of the flood − GT range in mirrored lateral bands
# (multipath is symmetric about the fold, a turn is not). Its range offset is a property of the camera,
# not of the angle: per angle the largest shift with the flood behind the GT in every band (MPI only
# lengthens), and the smallest of these over all angles (≈ the widest corner, least multipath) is applied
# to every angle, so narrow corners keep their multipath visible.
GT_OF = {s: s for s in [*PATTERNS, FLOOD]}
GT_TURN = {"10x10": True, "33x33": False, FLOOD: True}   # series whose GT may turn about the fold
TURN_GRID = np.radians(np.arange(-4.0, 4.0001, 0.02))   # [rad] searched turns
FLOOD_BAND_W = 0.05      # [m] lateral bands for the flood


def turned_frame(sc, phi, s=1.0):
    # Scene corner turned by phi about the fold axis (through O along e_y), then scaled about the camera.
    a, O = sc["R"][1], sc["O"]
    K = np.array([[0, -a[2], a[1]], [a[2], 0, -a[0]], [-a[1], a[0], 0]])
    Rm = np.eye(3) + np.sin(phi) * K + (1 - np.cos(phi)) * K @ K
    walls = [(Rm @ n, -(Rm @ n) @ O * s) for n, _ in sc["walls"]]
    fr = corner_frame(walls)
    fr["mx"] = lateral_margin(fr["corner"])
    return fr


def range_ratio(P, fr):
    return np.linalg.norm(P, axis=1) / ray_range_to_corner(P, fr["walls"])


def fit_sl(P, sc, turn):
    # (phi, s) of a pattern's SL points: wall points away from the fold, robust (median abs residual).
    far = np.abs(sc["to_corner"](P)[:, 0]) > sc["mx"]
    P = P[far]
    best = (np.inf, 0.0, 1.0)
    for phi in (TURN_GRID if turn else [0.0]):
        k = range_ratio(P, turned_frame(sc, phi))
        k = k[np.isfinite(k) & (np.abs(k - 1) < 0.1)]
        if len(k) < MIN_WALL_POINTS:
            continue
        s = float(np.median(k))
        best = min(best, (float(np.median(np.abs(k - s))), phi, s))
    return best[1], best[2]


def flood_bands(k, x, cap):
    # Median range ratio per lateral band, symmetric about the fold (NaN where too few points).
    edges = np.linspace(-cap, cap, 2 * int(np.ceil(cap / FLOOD_BAND_W)) + 1)
    return np.array([np.median(k[m]) if (m := (x >= lo) & (x < hi)).sum() > 200 else np.nan
                     for lo, hi in zip(edges[:-1], edges[1:])])


print("\nPer-series GT = scene corner turned by φ about the fold, shifted (s − 1)·d at the fold")
FLOOD_TOUCH, FLOOD_TURN = {}, {}
for angle in ANGLES:
    sc = SCENE[angle]
    row = []
    for pat in PATTERNS:
        if pat == "33x33" and pat not in sources:
            PGT[(angle, pat)] = sc
            continue
        phi, s = fit_sl(sl_points(pat, angle), sc, GT_TURN[pat])
        PGT[(angle, pat)] = turned_frame(sc, phi, s)
        row.append(f"{pat} φ {np.degrees(phi):+5.2f}° {(s - 1) * sc['dist'] * 1e3:+5.1f} mm")
    fl = FLOOD_MAP[angle].reshape(-1, 3)
    fl = fl[np.isfinite(fl).all(axis=1)]
    x, y = sc["to_corner"](fl)[:, :2].T
    cap = WALL_LENGTH * np.sin(np.radians(inner_angle(sc["corner"]) / 2))
    on = (np.abs(y) < 0.25) & (np.abs(x) < cap)
    fl, x = fl[on], x[on]
    best = (np.inf, 0.0, None)
    for phi in (TURN_GRID if GT_TURN[FLOOD] else [0.0]):
        k = range_ratio(fl, turned_frame(sc, phi))
        ok = np.abs(k - 1) < 0.15
        med = flood_bands(k[ok], x[ok], cap)
        asym = med - med[::-1]
        if np.isfinite(asym).sum() >= 4:
            best = min(best, (float(np.nanmean(asym ** 2)), phi, med), key=lambda b: b[0])
    FLOOD_TURN[angle] = best[1]
    FLOOD_TOUCH[angle] = (float(np.nanmin(best[2])) - 1) * sc["dist"]     # [m] at the fold
    print(f"  {angle}°: " + " | ".join(row) + f" | flood φ {np.degrees(best[1]):+5.2f}°, "
          f"touch {FLOOD_TOUCH[angle] * 1e3:+5.1f} mm")

FLOOD_OFFSET = min(FLOOD_TOUCH.values())     # [m] one flood range offset for the whole series
print(f"flood GT: shifted by {FLOOD_OFFSET * 1e3:+.1f} mm at the fold for every angle; flood − GT per band:")
for angle in ANGLES:
    PGT[(angle, FLOOD)] = turned_frame(SCENE[angle], FLOOD_TURN[angle], 1 + FLOOD_OFFSET / SCENE[angle]["dist"])
    print(f"  {angle}°: {fmt(flood_behind(PGT[(angle, FLOOD)]['walls'], angle))}")


In [ ]:
# ── Region of interest (image region) and the evaluation frame ──────────────
from scipy.spatial import cKDTree as _KD


def gt_hit(P, frame):
    # Where the pixel ray of every point hits the corner of `frame` (corner coordinates).
    P = np.asarray(P, float)
    rh = P / np.linalg.norm(P, axis=1, keepdims=True)
    return frame["to_corner"](rh * ray_range_to_corner(P, frame["walls"])[:, None])


def runs_from(good, start, step):
    # Outer end of the run of good bins that starts at the first good bin from `start` on.
    k = start
    while 0 <= k < len(good) and not good[k]:
        k += step
    if not (0 <= k < len(good)):
        return start
    while 0 <= k + step < len(good) and good[k + step]:
        k += step
    return k


def roi_limits(fl, frame):
    Q = gt_hit(fl, frame)
    flc = frame["to_corner"](fl)
    on_wall = dist_to_gt(flc[:, 0], flc[:, 2], frame["corner"]) < ROI_TOL
    mx = frame["mx"]
    # vertical: bins in y on the walls (mx < |x| < 2·mx), contiguous around 0
    e = np.arange(-Y_HALF, Y_HALF + 1e-9, 0.02)
    sel = (np.abs(Q[:, 0]) > mx) & (np.abs(Q[:, 0]) < 2 * mx)
    idx = np.digitize(Q[sel, 1], e) - 1
    good = np.array([(idx == k).sum() > 30 and on_wall[sel][idx == k].mean() > 0.8 for k in range(len(e) - 1)])
    k0 = len(good) // 2
    y_lim = (e[runs_from(good, k0, -1)], e[runs_from(good, k0 - 1, +1) + 1])
    # lateral: on each side, the run of good bins from the margin outwards
    e = np.arange(-1.2, 1.2001, 0.01)
    sel = (Q[:, 1] >= y_lim[0]) & (Q[:, 1] <= y_lim[1])
    idx = np.digitize(Q[sel, 0], e) - 1
    good = np.array([(idx == k).sum() > 30 and on_wall[sel][idx == k].mean() > 0.8 for k in range(len(e) - 1)])
    kl = int(np.searchsorted(e, -mx)) - 1
    kr = int(np.searchsorted(e, mx))
    x_lim = (e[runs_from(good, kl, -1)], e[runs_from(good, kr, +1) + 1])
    cap = WALL_LENGTH * np.sin(np.radians(inner_angle(frame["corner"]) / 2))   # walls end at WALL_LENGTH
    return (max(x_lim[0], -cap), min(x_lim[1], cap)), y_lim


# Pixel of a point = the lens ray closest to its direction.
_rx, _ry = RAYS[FLOOD]
_RAY_TREE = _KD(np.c_[_rx.ravel(), _ry.ravel()])


def pixel_of(P):
    P = np.asarray(P, float)
    return _RAY_TREE.query(np.c_[P[:, 0] / P[:, 2], P[:, 1] / P[:, 2]])[1]


SWEEP, CORNERS, ROI, ROI_MASK, N_ROI, EVAL, FRAME_OF = {}, {}, {}, {}, {}, {}, {}
for angle in ANGLES:
    g = GT[angle]
    Pm = FLOOD_MAP[angle]
    flat = Pm.reshape(-1, 3)
    ok = np.isfinite(flat).all(axis=1)
    ROI[angle] = roi_limits(flat[ok], g)
    (x0, x1), (y0, y1) = ROI[angle]
    mask = np.zeros(len(flat), bool)
    Q = gt_hit(flat[ok], g)
    mask[np.flatnonzero(ok)] = (Q[:, 0] >= x0) & (Q[:, 0] <= x1) & (Q[:, 1] >= y0) & (Q[:, 1] <= y1)
    ROI_MASK[angle] = mask.reshape(Pm.shape[:2])
    N_ROI[angle] = int(mask.sum())
    EVAL[angle] = PGT[(angle, "33x33")]            # axes / labels of the plots

    series_pts = {(FLOOD, "ToF"): flat[mask]}
    for pat in PATTERNS:
        for a in (1, 2, 3):
            P = RAW[(angle, pat, a)]
            series_pts[(pat, a)] = P[mask[pixel_of(P)]] if len(P) else np.zeros((0, 3))
    for (series, a), P in series_pts.items():
        fr = PGT[(angle, GT_OF[series])]
        FRAME_OF[(angle, series, a)] = fr
        SWEEP[(angle, series, a)] = fr["to_corner"](P) if len(P) else np.zeros((0, 3))
        CORNERS[(angle, series, a)] = fr["corner"]
    print(f"{angle}°: ROI {N_ROI[angle]} pixels (on the flood walls x = {x0:+.2f} … {x1:+.2f} m, "
          f"y = {y0:+.2f} … {y1:+.2f} m); in ROI A1/A2/A3: "
          + ", ".join(f"{p} " + "/".join(str(len(SWEEP[(angle, p, a)])) for a in (1, 2, 3)) for p in PATTERNS)
          + f", flood {len(SWEEP[(angle, FLOOD, 'ToF')])}")

print("\nGT wall range along the ray − point range, points farther than the margin from the fold "
      "(median ± MAD [mm]; + = point in front of the GT, [share > 10 mm in front]). Multipath only "
      "lengthens the ToF range, so ToF series should not lie in front of the scene GT:")
for angle in ANGLES:
    row = []
    for series, a in [(p, a) for p in PATTERNS for a in (1, 2, 3)] + [(FLOOD, "ToF")]:
        fr = FRAME_OF[(angle, series, a)]
        Pc = SWEEP[(angle, series, a)]
        P = (Pc - np.array([0.0, 0.0, fr["dist"]])) @ fr["R"] + fr["O"]          # back to the camera frame
        far = np.abs(Pc[:, 0]) > fr["mx"]
        dd = ray_range_to_corner(P[far], fr["walls"]) - np.linalg.norm(P[far], axis=1)
        dd = dd[np.isfinite(dd)]
        lab = f"{series} {'A' + str(a) if a != 'ToF' else 'ToF'}"
        row.append(f"{lab} {np.median(dd) * 1e3:+6.1f}±{np.median(np.abs(dd - np.median(dd))) * 1e3:4.1f} "
                   f"[{np.mean(dd > 0.01):4.0%}]" if len(dd) else f"{lab} –")
    print(f"  {angle}°: " + " | ".join(row[:3]) + "\n        " + " | ".join(row[3:6]) + "\n        " + row[6])


## Results — plot functions

The plots of `MPIsweep.ipynb`, with the corner angle in place of the distance (one row per corner):

1. **Top view** of the corner in the corner frame, all approaches × all illuminations, seven columns:
   10x10 A1 | A2 | A3, 33x33 A1 | A2 | A3, flood (iToF only). Black line: fitted GT corner. The title
   gives the RMSE of the perpendicular distance to the GT walls and the number of points. All panels
   in a row share the axes.
2. **Profile across the fold.** Depth error z − z_GT(x) over the lateral position relative to the
   fold. A correctly reproduced kink is a flat band around 0 up to the fold; multipath is a bump that
   grows towards the fold. For the flood cloud the black line is the median per 1 cm bin.
3. **Summary over the corner angle.** (a) RMSE to the GT walls, (b) mean signed depth error in the
   fold region (|x| < `FOLD_HALF_WIDTH`) — the multipath bump, (c) valid dots per approach as a share
   of the calibrated trails (flood: share of the ROI pixels with a return). 10x10 solid, 33x33 dashed,
   flood black. Below: the same numbers as a table.

In [ ]:
# Okabe–Ito, same assignment as noiseEstimation.ipynb / MPIsweep.ipynb.
A_COLOR = {1: "#0072B2", 2: "#D55E00", 3: "#009E73", "ToF": "#000000"}
A_NAME = {1: "A1 consistency", 2: "A2 ML fusion", 3: "A3 active brightness", "ToF": "iToF only"}
COLUMNS = [(pat, a) for pat in PATTERNS for a in (1, 2, 3)] + [(FLOOD, "ToF")]
COL_LABEL = {**{(pat, a): f"{PATTERNS[pat]['label']} · {A_NAME[a]}" for pat in PATTERNS for a in (1, 2, 3)},
             (FLOOD, "ToF"): "Flood · iToF only"}
FLOOD_PLOT_MAX = 20000        # flood points drawn per panel (random subset)
PROFILE_YLIM = 0.06           # ±[m]
TOPVIEW_Z_EXTRA = {("MPI_1m_WW", 60, "33x33"): 0.08}   # [m] extra headroom above (folder, angle, pattern)
FOLD_HALF_WIDTH = 0.05        # [m]
_rng = np.random.default_rng(0)


def series_stats(angle, series, a):
    pts = SWEEP[(angle, series, a)]
    if len(pts) == 0:
        return pts, np.nan
    err = dist_to_gt(pts[:, 0], pts[:, 2], CORNERS[(angle, series, a)])
    return pts, float(np.sqrt(np.mean(err ** 2)))


def view_limits(angle):
    fl = SWEEP[(angle, FLOOD, "ToF")]
    d = EVAL[angle]["dist"]
    # upper limit: fold, flood and every dot series of the row (99.5th percentile, ignores single outliers)
    z_top = [float(np.percentile(SWEEP[(angle, s, a)][:, 2], 99.5)) for s, a in COLUMNS if len(SWEEP[(angle, s, a)])]
    z_hi = max([d] + z_top) + 0.05
    return (fl[:, 0].min() - 0.05, fl[:, 0].max() + 0.05), (fl[:, 2].min() - 0.05, z_hi)


def plot_topview():
    fig, axs = plt.subplots(len(ANGLES), len(COLUMNS), figsize=(3.1 * len(COLUMNS), 2.9 * len(ANGLES)),
                            squeeze=False)
    for i, angle in enumerate(ANGLES):
        xlim, zlim = view_limits(angle)
        for j, (series, a) in enumerate(COLUMNS):
            ax = axs[i, j]
            pts, rmse = series_stats(angle, series, a)
            if series == FLOOD and len(pts) > FLOOD_PLOT_MAX:
                pts = pts[_rng.choice(len(pts), FLOOD_PLOT_MAX, replace=False)]
            ax.scatter(pts[:, 0], pts[:, 2], s=1.5 if series == FLOOD else 5, lw=0,
                       c=A_COLOR[a], alpha=0.25 if series == FLOOD else 0.8)
            c = CORNERS[(angle, series, a)]
            ax.plot(c[:, 0], c[:, 1], "k-", lw=1.2)
            extra = TOPVIEW_Z_EXTRA.get((MEAS_ROOT.name, angle, series), 0.0)
            ax.set_xlim(xlim); ax.set_ylim(zlim[0], zlim[1] + extra)
            ax.set_aspect("equal", adjustable="datalim")   # true angles: same scale in x and z
            if extra:
                ax.apply_aspect()                           # datalim may trim z again — keep the headroom
                lo, hi = ax.get_ylim()
                if hi < zlim[1] + extra:
                    ax.set_ylim(lo + (zlim[1] + extra - hi), zlim[1] + extra)
            ax.set_title(f"RMSE {rmse * 1e3:.1f} mm · n={len(SWEEP[(angle, series, a)])}", fontsize=8)
            ax.tick_params(labelsize=7)
            ax.grid(alpha=0.25, lw=0.5)
            if i == 0:
                ax.annotate(COL_LABEL[(series, a)], (0.5, 1.25), xycoords="axes fraction",
                            ha="center", fontsize=9, fontweight="bold")
            if j == 0:
                ax.set_ylabel(f"{angle}° (GT {inner_angle(EVAL[angle]['corner']):.0f}°)\nz [m]", fontsize=9)
            if i == len(ANGLES) - 1:
                ax.set_xlabel("x [m]", fontsize=8)
    fig.suptitle(f"Real corners · top view (x–z, corner frame) · black: GT per series (scene corner, turned + scaled)", y=1.0, fontsize=13)
    fig.tight_layout()
    plt.show()


def plot_profile():
    fig, axs = plt.subplots(len(ANGLES), len(COLUMNS), figsize=(3.1 * len(COLUMNS), 2.4 * len(ANGLES)),
                            squeeze=False, sharey=True)
    for i, angle in enumerate(ANGLES):
        xlim, _ = view_limits(angle)
        for j, (series, a) in enumerate(COLUMNS):
            ax = axs[i, j]
            c = CORNERS[(angle, series, a)]
            pts = SWEEP[(angle, series, a)]
            x_rel = pts[:, 0] - c[1, 0]
            dz = pts[:, 2] - z_gt(pts[:, 0], c)
            if series == FLOOD:
                sub = _rng.choice(len(pts), min(len(pts), FLOOD_PLOT_MAX), replace=False)
                ax.scatter(x_rel[sub], dz[sub], s=1, lw=0, c="0.6", alpha=0.3)
                bins = np.arange(x_rel.min(), x_rel.max() + 0.01, 0.01)
                idx = np.digitize(x_rel, bins)
                med = [np.nanmedian(dz[idx == k]) if np.sum(np.isfinite(dz[idx == k])) > 20 else np.nan
                       for k in range(1, len(bins))]
                ax.plot(bins[:-1] + 0.005, med, "k-", lw=1.5)
            else:
                ax.scatter(x_rel, dz, s=6, lw=0, c=A_COLOR[a], alpha=0.85)
            ax.axhline(0, color="k", lw=0.8)
            ax.axvline(0, color="k", lw=0.6, ls=":")
            ax.set_ylim(-PROFILE_YLIM, PROFILE_YLIM)
            ax.set_xlim(xlim[0] - c[1, 0], xlim[1] - c[1, 0])
            ax.tick_params(labelsize=7)
            ax.grid(alpha=0.25, lw=0.5)
            if i == 0:
                ax.annotate(COL_LABEL[(series, a)], (0.5, 1.12), xycoords="axes fraction",
                            ha="center", fontsize=9, fontweight="bold")
            if j == 0:
                ax.set_ylabel(f"{angle}°\nz − z_GT [m]", fontsize=9)
            if i == len(ANGLES) - 1:
                ax.set_xlabel("x − x_fold [m]", fontsize=8)
    fig.suptitle("Real corners · depth error across the fold (dotted: fold position)", y=1.0, fontsize=13)
    fig.tight_layout()
    plt.show()


def plot_summary():
    rows = []
    for series, a in COLUMNS:
        for angle in ANGLES:
            pts, rmse = series_stats(angle, series, a)
            c = CORNERS[(angle, series, a)]
            near = np.abs(pts[:, 0] - c[1, 0]) < FOLD_HALF_WIDTH if len(pts) else np.zeros(0, bool)
            fold_err = float(np.mean(pts[near, 2] - z_gt(pts[near, 0], c))) if near.any() else np.nan
            n_ref = N_TRAILS.get(series) or N_ROI[angle]
            rows.append((series, a, angle, rmse, fold_err, len(pts) / n_ref, int(near.sum())))
    rows_arr = np.array([(r[2], r[3], r[4], r[5]) for r in rows], float)

    fig, axs = plt.subplots(1, 3, figsize=(17, 4.6))
    for series, a in COLUMNS:
        sel = [k for k, r in enumerate(rows) if r[0] == series and r[1] == a]
        ls = "--" if series == "33x33" else "-"
        lw = 2.4 if series == FLOOD else 1.7
        for ax, col, scale in zip(axs, (1, 2, 3), (1e3, 1e3, 100)):
            ax.plot(rows_arr[sel, 0], rows_arr[sel, col] * scale, ls, color=A_COLOR[a], lw=lw,
                    marker="o", ms=4, label=COL_LABEL[(series, a)])
    axs[0].set_ylabel("RMSE to GT walls [mm]")
    axs[1].set_ylabel(f"mean z − z_GT, |x − x_fold| < {FOLD_HALF_WIDTH * 100:.0f} cm [mm]")
    axs[1].axhline(0, color="k", lw=0.8)
    axs[2].set_ylabel("valid [% of trails / ROI pixels]")
    for ax, t in zip(axs, ("(a) accuracy", "(b) fold error (multipath bump)", "(c) coverage")):
        ax.set_xlabel("inner corner angle [deg]")
        ax.set_xticks(ANGLES)
        ax.set_title(t)
        ax.grid(alpha=0.3)
    axs[2].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.02, 0.5))
    fig.suptitle("Real corners · summary over the corner angle", fontsize=13)
    fig.tight_layout()
    plt.show()

    print(f"{'series':28s}" + "".join(f"{a:>8d}°" for a in ANGLES) + "   (RMSE [mm] / fold error [mm])")
    for series, a in COLUMNS:
        sel = [r for r in rows if r[0] == series and r[1] == a]
        print(f"{COL_LABEL[(series, a)]:28s}" + "".join(f"{r[3] * 1e3:9.1f}" for r in sel))
        print(f"{'':28s}" + "".join(f"{r[4] * 1e3:+9.1f}" for r in sel))

## Results

One row per corner, from the narrowest (strongest interreflection) to the widest (90°, the
`MPI_45deg` corner of the simulation). Every series is compared with its own GT: the scene corner (10x10 + tilt-corrected 33x33
triangulation), scaled to that series, so a constant offset of a whole column is the offset of that series to the scene
(table "GT wall − point range" above), and the change towards the fold is the multipath.


In [ ]:
plot_topview()
plot_profile()
plot_summary()

## Export for `MPI_flood_reflectivity.ipynb`

The flood clouds (after SOR), the region of interest, the flood walls and the 33x33 corner of every
angle are written to `Results/mpi_sweep_real/flood_export_<folder>.pkl`, so the reflectivity fit of the
simulation can work on exactly the clouds evaluated here without rerunning the sweep.

In [ ]:
import pickle
FLOOD_EXPORT = CACHE_DIR / f"flood_export_{MEAS_ROOT.name}.pkl"
with open(FLOOD_EXPORT, "wb") as f:
    pickle.dump(dict(
        meas_root=str(MEAS_ROOT), frame_mode=FRAME_MODE, sor=SOR_TAG, angles=list(ANGLES),
        tape=FOLD_DISTANCE.get(MEAS_ROOT.name),
        flood={a: FLOOD_MAP[a] for a in ANGLES},
        roi={a: tuple(map(tuple, ROI[a])) for a in ANGLES},
        u_fold={a: GT[a]["u_fold"] for a in ANGLES},
        flood_walls={a: [(n.tolist(), float(d)) for n, d in GT[a]["walls"]] for a in ANGLES},
        walls_33x33={a: [(n.tolist(), float(d)) for n, d in PGT[(a, "33x33")]["walls"]] for a in ANGLES},
        walls_10x10={a: [(n.tolist(), float(d)) for n, d in PGT[(a, "10x10")]["walls"]] for a in ANGLES},
    ), f)
print(f"flood export → {FLOOD_EXPORT}")

## Notes

- Data: AandB, 100 MHz. `FRAME_MODE = "first"` (default) evaluates the first RAW frame of every set
  (`ToF_PCD/<frame>.pcd`, `SL/<frame>.exr` next to the RAWs); `"avg"` averages the 10 frames tap by tap
  (`process_2rad(Diffs=…)` keeps the sub-LSB part) into `avg.pcd` / `avg.exr`.
- Depth calibration: 10x10 and flood with `CalibData_Julian.hdf5`, 33x33 with
  `CalibData_Julian_LensOnly.hdf5` — the combinations that reproduce the calibration PCDs of 25.08
  (10x10, delay 7/0/5) and 30.09 (33x33, delay 0/0/0) to ~1 cm. With the other file the range is off
  by ≈ 0.18 m.
- Dot calibrations are the existing ones, selected by `CAL_VARIANT`: `corr13456` (6-stage ToF error
  model, refitted here and applied to the test clouds) or `lpc` (light-path correction only). Like the
  calibrations, every test cloud is SOR-filtered (6 neighbours, 0.4 σ) before the light-path
  correction and the error model. The 10x10 calibration data dates from 25.08.
- The flood (VCSEL 4) has no calibration of its own: no light-path correction, and its absolute range
  carries an unknown constant offset. The two dot calibrations also disagree on the absolute depth
  (table in the ground-truth step).
- Ground truth: one scene corner per angle from the SL triangulation alone (no ToF depth): first the
  10x10, after the tilt step refitted with the tilt-corrected 33x33 dots (see the ground-truth step); the projector positions do not enter it. At 1 m (tape 1.03 m, reference point on
  the camera unknown) it puts the fold at ≈ 1.03–1.07 m; the 33x33 triangulation says ≈ 1.15 m in this
  session (calibration of 30.09 does not fit 07.10, see above).
- v11: depth calibrations `CalibData_10x10_v1.1` / `CalibData_Coherent_v1.1` (dot sweeps), SL
  calibrations from `noiseEstimation_v11_*`, no light-path correction, no error model. The flood keeps
  `CalibData_Julian.hdf5` and its existing PCDs.
- 100 MHz → unambiguous up to 1.5 m. The background behind the corner wraps; everything beyond
  `Z_MAX` and outside the ROI is dropped. The folds lie at ≈ 1.3–1.5 m, so the indirect paths of the
  interreflection are longer than 1.5 m and wrap as well: their phasors pull the mixed phase towards
  *shorter* ranges. The flood corner is therefore rounded towards the camera here, unlike a
  multipath bulge that stays within the unambiguity range.
- The approaches' global settings (`EPS_THRESHOLD`, peak finding, …) are the defaults of
  `approaches.py`, unchanged.